# Cross-Sensitivity of the Downstream-Rate Ratio and the Parallel-Branch-Imbalance Ratio (Corrected)

This notebook supersedes `cross_sensitivity_dsr_pbi_sweep.ipynb`, which used
an incorrect `--pbi-ratio` implementation (it compared only the single node
immediately before a join, not the whole parallel branch). `--pbi-ratio` now
enforces the cumulative fork-to-join branch-SUM ratio — see
`MILP/finn_milp.md`'s "Rate coherence" section and its 2026-09-29 History
entries for the full correction. All data here was solved with the
corrected constraint.

Four sweep families, all on the real S12-dense `nearest_upsample` 512x512
architecture, at the same hard resource budget as the reference
mixed-precision solve (LUT 50% / BRAM 50% / DSP 90%, `--force-dsp`,
`--candidate-bits 4,6,8`):

- **Sweep A** — `--pbi-ratio` held fixed at 1.5 (confirmed binding-and-
  feasible), `--dsr-ratio` swept over {unconstrained, 5.0, 3.0, 1.5}. Does an
  *enforced* tight `--pbi-ratio` change how `--dsr-ratio` behaves as it
  tightens?
- **Sweep B** — `--dsr-ratio` held fixed at 1.5, `--pbi-ratio` swept over
  {unconstrained, 3.0, 2.0, 1.5}. The mirror question.
- **Sweep C** — `--pbi-ratio` left **fully unconstrained** throughout,
  `--dsr-ratio` swept alone over {unconstrained, 5.0, 3.0, 1.5}. Shows how
  BOTH diagnostics (`branch_imbalance` for parallel-branch balance,
  `chain_rate_imbalance` for chain coherence) respond to `--dsr-ratio` in
  isolation, with `--pbi-ratio` never touched at all.
- **Sweep D** — `--dsr-ratio` left **fully unconstrained** throughout,
  `--pbi-ratio` swept alone over {unconstrained, 3.0, 2.0, 1.5}. The mirror
  of Sweep C.

Sweeps C and D are the direct answer to "how does pbi vary with dsr, and
vice versa, under otherwise-fixed constraints" — they isolate each ratio's
effect on the OTHER's natural, unconstrained diagnostic value, with nothing
else in play. Sweeps A and B instead ask whether an *actively enforced*
tight value on one changes the other's own enforced behavior as it tightens
-- a related but distinct question.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Academic presentation: bold titles/axis labels, generous sizing, clean grid.
plt.rcParams.update({
    "font.size": 12,
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.labelweight": "bold",
    "axes.labelsize": 12,
    "legend.fontsize": 11,
    "legend.title_fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.dpi": 130,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "MILP" / "finn_milp.py").is_file():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("Could not locate repo root (MILP/finn_milp.py not found in any ancestor).")
    REPO_ROOT = REPO_ROOT.parent

SWEEP_DIR = REPO_ROOT / "MILP" / "artifacts" / "S12_dense_nearest_upsample_512_v1_cross_sensitivity_v2"
FIG_DIR = REPO_ROOT / "analysis" / "report" / "MILP" / "figures_v2"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Sweep dir:", SWEEP_DIR)


In [ ]:
def load_run(tag: str) -> dict:
    """One row of diagnostics for a single MILP solve, keyed by tag."""
    json_path = SWEEP_DIR / tag / f"layer_bits_folding_{tag}.json"
    with open(json_path) as f:
        d = json.load(f)
    diag = d["_diagnostics"]
    bi = diag.get("branch_imbalance", {}) or {}
    cri = diag.get("chain_rate_imbalance", {}) or {}
    wbits = d.get("layer_weight_bits") or {}
    abits = d.get("layer_act_bits") or {}
    return {
        "tag": tag,
        "dsr_ratio": diag.get("dsr_ratio"),
        "pbi_ratio": diag.get("pbi_ratio"),
        "status": d["status"],
        "avg_weight_bits": sum(wbits.values()) / len(wbits) if wbits else None,
        "avg_act_bits": sum(abits.values()) / len(abits) if abits else None,
        "lut_pct": diag.get("lut_pct_of_budget"),
        "bram_pct": diag.get("bram_pct_of_budget"),
        "dsp_pct": diag.get("dsp_pct_of_budget"),
        "total_cycles": diag.get("total_cycles"),
        "bi_median_ratio": bi.get("median_ratio"),
        "bi_max_ratio": bi.get("max_ratio"),
        "cri_median_ratio": cri.get("median_ratio"),
        "cri_max_ratio": cri.get("max_ratio"),
    }


SWEEP_A_TAGS = ["sweepA_pbiFix1.5_dsr_off", "sweepA_pbiFix1.5_dsr5.0", "sweepA_pbiFix1.5_dsr3.0", "sweepA_pbiFix1.5_dsr1.5"]
SWEEP_B_TAGS = ["sweepB_dsrFix1.5_pbi_off", "sweepB_dsrFix1.5_pbi3.0", "sweepB_dsrFix1.5_pbi2.0", "sweepB_dsrFix1.5_pbi1.5"]
SWEEP_C_TAGS = ["both_off", "sweepC_pbiOff_dsr5.0", "sweepC_pbiOff_dsr3.0", "sweepC_pbiOff_dsr1.5"]
SWEEP_D_TAGS = ["both_off", "sweepD_dsrOff_pbi3.0", "sweepD_dsrOff_pbi2.0", "sweepD_dsrOff_pbi1.5"]

DSR_LABELS = ["Unconstrained", "5.0", "3.0", "1.5"]
PBI_LABELS_B = ["Unconstrained", "3.0", "2.0", "1.5"]
PBI_LABELS_D = ["Unconstrained", "3.0", "2.0", "1.5"]

all_tags = sorted(set(SWEEP_A_TAGS + SWEEP_B_TAGS + SWEEP_C_TAGS + SWEEP_D_TAGS))
rows = [load_run(tag) for tag in all_tags]
df = pd.DataFrame(rows).set_index("tag")
df.to_csv(SWEEP_DIR / "cross_sensitivity_v2_summary.csv")

df_a = df.loc[SWEEP_A_TAGS].copy(); df_a["x_label"] = DSR_LABELS
df_b = df.loc[SWEEP_B_TAGS].copy(); df_b["x_label"] = PBI_LABELS_B
df_c = df.loc[SWEEP_C_TAGS].copy(); df_c["x_label"] = DSR_LABELS
df_d = df.loc[SWEEP_D_TAGS].copy(); df_d["x_label"] = PBI_LABELS_D

df


## Sweep A: Enforced Join-Balance Ratio Fixed at 1.5, Downstream-Rate Ratio Tightening

Does holding `--pbi-ratio` at a real, binding value change how `--dsr-ratio`
behaves as it tightens?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(df_a["x_label"], df_a["cri_median_ratio"], marker="o", linewidth=2.5, markersize=8, label="Chain-Rate Imbalance (median)")
axes[0].plot(df_a["x_label"], df_a["bi_median_ratio"], marker="s", linewidth=2.5, markersize=8, label="Branch Imbalance (median)")
axes[0].axhline(1.5, color="grey", linestyle="--", linewidth=1.2, label="Fixed Join-Balance Ratio (1.5)")
axes[0].set_title("Achieved Imbalance Ratios as the\nDownstream-Rate Ratio Tightens")
axes[0].set_xlabel("Downstream-Rate Ratio (Join-Balance Ratio Fixed at 1.5)")
axes[0].set_ylabel("Median Achieved Ratio")
axes[0].legend(title="Diagnostic", frameon=True)

axes[1].plot(df_a["x_label"], df_a["dsp_pct"], marker="^", linewidth=2.5, markersize=8, color="#2E7D32", label="DSP")
axes[1].plot(df_a["x_label"], df_a["lut_pct"], marker="o", linewidth=2.5, markersize=8, color="#1565C0", label="LUT")
axes[1].plot(df_a["x_label"], df_a["bram_pct"], marker="s", linewidth=2.5, markersize=8, color="#EF6C00", label="BRAM18K")
axes[1].set_title("Resource Utilization as the\nDownstream-Rate Ratio Tightens")
axes[1].set_xlabel("Downstream-Rate Ratio (Join-Balance Ratio Fixed at 1.5)")
axes[1].set_ylabel("Utilization (% of Device Budget)")
axes[1].legend(title="Resource", frameon=True)
fig.tight_layout()
fig.savefig(FIG_DIR / "sweepA_summary.png", bbox_inches="tight")
plt.show()


## Sweep B: Enforced Downstream-Rate Ratio Fixed at 1.5, Join-Balance Ratio Tightening

The mirror of Sweep A.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(df_b["x_label"], df_b["bi_median_ratio"], marker="o", linewidth=2.5, markersize=8, label="Branch Imbalance (median)")
axes[0].plot(df_b["x_label"], df_b["cri_median_ratio"], marker="s", linewidth=2.5, markersize=8, label="Chain-Rate Imbalance (median)")
axes[0].axhline(1.5, color="grey", linestyle="--", linewidth=1.2, label="Fixed Downstream-Rate Ratio (1.5)")
axes[0].set_title("Achieved Imbalance Ratios as the\nJoin-Balance Ratio Tightens")
axes[0].set_xlabel("Join-Balance Ratio (Downstream-Rate Ratio Fixed at 1.5)")
axes[0].set_ylabel("Median Achieved Ratio")
axes[0].legend(title="Diagnostic", frameon=True)

axes[1].plot(df_b["x_label"], df_b["dsp_pct"], marker="^", linewidth=2.5, markersize=8, color="#2E7D32", label="DSP")
axes[1].plot(df_b["x_label"], df_b["lut_pct"], marker="o", linewidth=2.5, markersize=8, color="#1565C0", label="LUT")
axes[1].plot(df_b["x_label"], df_b["bram_pct"], marker="s", linewidth=2.5, markersize=8, color="#EF6C00", label="BRAM18K")
axes[1].set_title("Resource Utilization as the\nJoin-Balance Ratio Tightens")
axes[1].set_xlabel("Join-Balance Ratio (Downstream-Rate Ratio Fixed at 1.5)")
axes[1].set_ylabel("Utilization (% of Device Budget)")
axes[1].legend(title="Resource", frameon=True)
fig.tight_layout()
fig.savefig(FIG_DIR / "sweepB_summary.png", bbox_inches="tight")
plt.show()


## Sweep C: Join-Balance Ratio Left Unconstrained, Downstream-Rate Ratio Swept Alone

`--pbi-ratio` is never set here. Does tightening `--dsr-ratio` by itself move
the branch-imbalance diagnostic as a side effect?

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(df_c["x_label"], df_c["cri_median_ratio"], marker="o", linewidth=2.5, markersize=8, label="Chain-Rate Imbalance (median)")
ax.plot(df_c["x_label"], df_c["bi_median_ratio"], marker="s", linewidth=2.5, markersize=8, label="Branch Imbalance (median)")
ax.set_title("Both Diagnostics' Response to the\nDownstream-Rate Ratio Alone (Join-Balance Unconstrained)")
ax.set_xlabel("Downstream-Rate Ratio (Join-Balance Ratio Never Set)")
ax.set_ylabel("Median Achieved Ratio")
ax.legend(title="Diagnostic", frameon=True)
fig.tight_layout()
fig.savefig(FIG_DIR / "sweepC_summary.png", bbox_inches="tight")
plt.show()


## Sweep D: Downstream-Rate Ratio Left Unconstrained, Join-Balance Ratio Swept Alone

`--dsr-ratio` is never set here. Does tightening `--pbi-ratio` by itself move
the chain-rate-imbalance diagnostic as a side effect?

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(df_d["x_label"], df_d["bi_median_ratio"], marker="o", linewidth=2.5, markersize=8, label="Branch Imbalance (median)")
ax.plot(df_d["x_label"], df_d["cri_median_ratio"], marker="s", linewidth=2.5, markersize=8, label="Chain-Rate Imbalance (median)")
ax.set_title("Both Diagnostics' Response to the\nJoin-Balance Ratio Alone (Downstream-Rate Unconstrained)")
ax.set_xlabel("Join-Balance Ratio (Downstream-Rate Ratio Never Set)")
ax.set_ylabel("Median Achieved Ratio")
ax.legend(title="Diagnostic", frameon=True)
fig.tight_layout()
fig.savefig(FIG_DIR / "sweepD_summary.png", bbox_inches="tight")
plt.show()


## Interpretation

_Filled in after the sweep completes, from the real numbers in the table and
figures above -- see the accompanying chat message for the final reading._